# 01 · Part 1: exact baseline

Exact search with FAISS (`IndexFlatL2`) is the reference for every other result. This notebook does two things:
1. checks that our exact neighbours agree with the neighbours stored in the ann-benchmarks files;
2. times exact search one query at a time, and with all queries in one batch.

Our recall@10 function and its unit tests live in `00_core.ipynb` (section *Recall@10*); the tests run again every time that notebook is loaded below.

In [1]:
%run ./00_core.ipynb

.

.

.

.

.

...

                                                                                         [100%]


.

.

                                                                                           [100%]


mode=full  results -> results  timing CPU 2  build threads 12


## Agreement with the stored neighbours
For each query we compare the stored top 10 with our exact top 10 using the tie-aware recall@10, in both directions. We also count queries whose two top-10 id sets are identical, and compare the stored distances with ours. FAISS works with squared L2. We believe the files store plain L2 for SIFT and 1 − cos for GloVe; on unit vectors 1 − cos equals squared L2 ÷ 2. The largest difference below shows whether those conversions are right.

The stored neighbours are only valid for the full base set, so this check always loads every base vector, even in quick mode. Quick mode checks only the first `agreement_queries` queries.

In [2]:
def in_stored_units(name, squared_l2):
    """Our squared L2 distances in the units the HDF5 file stores: L2 for SIFT, 1 - cos for (unit-norm) GloVe."""
    return np.sqrt(squared_l2) if name == "sift" else squared_l2 / 2


def agreement(name, data, n_queries):
    queries = data.queries[:n_queries]
    ours = ground_truth(name, data.base, queries)
    stored = data.stored_neighbors[:n_queries].astype(np.int64)
    stored_vs_ours = recall_at_k(stored[:, :K], data.base, queries, ours, per_query=True)
    ours_vs_stored = recall_at_k(ours[:, :K], data.base, queries, stored, per_query=True)
    squared = ((data.base[stored[:, :K]].astype(np.float64) - queries.astype(np.float64)[:, None, :]) ** 2).sum(-1)
    difference = np.abs(in_stored_units(name, squared) - data.stored_distances[:n_queries, :K])
    return dict(
        method="agreement", split="all queries" if n_queries == len(data.queries) else "tune", n_queries=n_queries,
        recall_at_10=float(stored_vs_ours.mean()),
        recall_ours_vs_stored=float(ours_vs_stored.mean()),
        queries_below_full_recall=int((stored_vs_ours < 1).sum()),
        identical_top10_share=float(np.mean([set(a) == set(b) for a, b in zip(ours[:, :K], stored[:, :K])])),
        max_abs_distance_diff=float(difference.max()),
        zero_norm_base_vectors=int((np.linalg.norm(data.base, axis=1) == 0).sum()),
    )

## Timing exact search
Both timings use the same protocol as every other QPS number:
- **One query at a time:** one pinned thread, median of `repeats` passes, over queries 5,000–5,999.
- **One batch:** all reporting-half queries in a single `search` call, timed twice: pinned to one thread, and with `BUILD_THREADS` unpinned threads.

Quick mode runs on the first `n_base` vectors, the same base set Part 2 uses in quick mode.

**Dimension check.** The single-thread batch is also timed with SIFT cut to its first 100 dimensions and GloVe zero-padded to 128 (padding leaves every distance unchanged). This shows how much of the batch speed comes from how FAISS's bundled BLAS handles d = 100 versus d = 128 (see `design_note.md`). These copies are used for timing only.

In [3]:
for name in ("sift", "glove"):
    full = load_dataset(name)
    check = agreement(name, full, CONFIG["agreement_queries"])
    check_row = {**run_header(1, name, full, "exact", {}, np.nan, np.nan), "param_name": "", "param_value": 0, **check}

    data = full if CONFIG["n_base"] is None else Dataset(name, full.base[: CONFIG["n_base"]].copy(), full.queries, None, None)
    del full
    gt = ground_truth(name, data.base, data.queries)

    [one_at_a_time] = benchmark_exact(1, name, data, gt)
    one_at_a_time["method"] = "one_at_a_time"

    index, build_time = build_index("exact", data.base)
    header = run_header(1, name, data, "exact", {}, build_time, index_bytes(index))
    batches = []
    for threads in (1, BUILD_THREADS):
        timing = time_batch(index, data.queries[REPORT], threads=threads)
        batches.append({**header, "method": "batch", "search_threads": threads, "param_name": "", "param_value": 0,
                        "split": "report", "n_queries": len(REPORT), "qps": timing["qps"],
                        "times_s": json.dumps([round(t, 6) for t in timing["times"]])})

    other_dim, how = (100, "first 100 dims") if data.dim == 128 else (128, "zero-padded to 128")
    resized = lambda x: np.ascontiguousarray(x[:, :other_dim] if other_dim < x.shape[1] else
                                             np.hstack([x, np.zeros((len(x), other_dim - x.shape[1]), np.float32)]))
    other = Dataset(name, resized(data.base), resized(data.queries), None, None)
    other_index, other_build = build_index("exact", other.base)
    timing = time_batch(other_index, other.queries[REPORT], threads=1)
    dim_check = {**run_header(1, name, other, "exact", {}, other_build, index_bytes(other_index), variant=how),
                 "method": "batch", "param_name": "", "param_value": 0, "split": "report", "n_queries": len(REPORT),
                 "qps": timing["qps"], "times_s": json.dumps([round(t, 6) for t in timing["times"]])}
    del other, other_index

    write_rows([check_row, one_at_a_time, *batches, dim_check], part=1, section=name)
    print(f"{name}: agreement recall {check['recall_at_10']:.4f}, one at a time {one_at_a_time['qps']:.1f} QPS")
    del data, index

sift: agreement recall 1.0000, one at a time 47.8 QPS


glove: agreement recall 1.0000, one at a time 53.4 QPS


## Summary
The tables below come from `results.csv`, drawn by `part1_assets` in `00_core`, which also writes them for the report. `05_report_assets.ipynb` calls the same function again.

In [4]:
results = read_results()
agreement_table, timing_table = part1_agreement_table(results), part1_timing_table(results)
for name, table in part1_assets(results).items():
    print(name)
    display(table)

part1_agreement


,Dataset,Queries,Recall: stored vs ours,Recall: ours vs stored,Same top-10,Max dist. diff.
0,SIFT,"10,000",1.000,1.000,99.4%,1.5e-05
1,GloVe-100,"10,000",1.000,1.000,100.0%,1.8e-07


part1_timing


,Dataset,Vectors,d,Dimension check,Method,Threads,QPS,ms/query,Speed-up
0,SIFT,"1,000,000",128,,one query at a time,1,47.8,20.939,1.0x
1,SIFT,"1,000,000",128,,one batch,1,156,6.407,3.3x
2,SIFT,"1,000,000",128,,one batch,12,773,1.294,16.2x
3,SIFT,"1,000,000",100,first 100 dims,one batch,1,194,5.167,4.1x
4,GloVe-100,"1,183,514",100,,one query at a time,1,53.4,18.737,1.0x
5,GloVe-100,"1,183,514",100,,one batch,1,164,6.116,3.1x
6,GloVe-100,"1,183,514",100,,one batch,12,762,1.312,14.3x
7,GloVe-100,"1,183,514",128,zero-padded to 128,one batch,1,132,7.575,2.5x


In [5]:
%%ipytest -qq

def test_exact_search_agrees_with_the_stored_neighbours():
    assert (agreement_table["recall_at_10"] > 0.99).all()


def test_every_dataset_has_all_three_timings():
    counts = timing_table.groupby("dataset")["method"].value_counts().unstack()
    assert (counts["one_at_a_time"] == 1).all() and (counts["batch"] == 3).all()

.

.

                                                                                           [100%]